# Phase 5: Random Forest Classifier Training & Evaluation

**Project:** Customer Churn Prediction & Retention Analytics  
**Model:** Balanced Random Forest Classifier (`n_estimators=200`, `class_weight='balanced'`, `random_state=42`, `n_jobs=-1`)  
**Preprocessed Input:** 19 approved features via `ColumnTransformer` (StandardScaler + OneHotEncoder)  
**Target:** `Churn Value` (0 = Retained, 1 = Churned)  

### Objectives:
1. Load the exact same 80/20 stratified train/test data splits used for Logistic Regression.
2. Build a scikit-learn `Pipeline` integrating preprocessing and Random Forest.
3. Fit the complete pipeline exclusively on the training data.
4. Generate test set predictions and predicted probabilities (`predict_proba`).
5. Evaluate classification metrics: Accuracy, Precision, Recall, F1-score, and ROC-AUC.
6. Plot and inspect the Confusion Matrix and ROC Curve.
7. Extract and visualize transformed feature importances (Gini importance).
8. Export the fitted pipeline to `models/random_forest_pipeline.joblib`.

## 1. Import Libraries

In [ ]:
import sys
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline

# Ensure project root is on Python path
sys.path.append("..")
from src.preprocess import get_train_test_data

fig_dir = Path("../outputs/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Stratified Train/Test Splits

In [ ]:
data = get_train_test_data(file_path="../data/processed/telco_churn_clean.csv")

X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
preprocessor = data["preprocessor"]

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")
print(f"y_train Shape: {y_train.shape} (Retained: {(y_train == 0).sum()}, Churned: {(y_train == 1).sum()})")
print(f"y_test Shape:  {y_test.shape} (Retained: {(y_test == 0).sum()}, Churned: {(y_test == 1).sum()})")

## 3. Build & Fit Random Forest Pipeline

In [ ]:
rf_classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", rf_classifier)
])

# Fit pipeline strictly on training data
pipeline.fit(X_train, y_train)
print("Random Forest Pipeline fitted successfully.")

## 4. Test Set Inference (Predictions & Probabilities)

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(f"Predictions generated for {len(y_pred):,} test instances.")

## 5. Performance Metrics & Classification Report

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print("RANDOM FOREST PERFORMANCE (TEST SET)")
print("=" * 60)
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f} ({prec*100:.2f}%)")
print(f"Recall:    {rec:.4f} ({rec*100:.2f}%)")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("=" * 60)
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

## 6. Confusion Matrix Analysis

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"True Negatives  (TN): {tn} (Retained correctly identified)")
print(f"False Positives (FP): {fp} (Retained misclassified as Churners)")
print(f"False Negatives (FN): {fn} (Churners missed by the model)")
print(f"True Positives  (TP): {tp} (Churners correctly identified)")
print(f"\nActual Churners in Test Set: {tp + fn}")
print(f"Detection Rate (Recall):      {(tp / (tp + fn)) * 100:.2f}%")
print(f"Miss Rate (False Negative):   {(fn / (tp + fn)) * 100:.2f}%")

# Visualize Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    [[tn, fp], [fn, tp]],
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Predicted Retained (0)", "Predicted Churned (1)"],
    yticklabels=["Actual Retained (0)", "Actual Churned (1)"],
    ax=ax,
    annot_kws={"size": 14, "weight": "bold"}
)
ax.set_title("Random Forest Confusion Matrix (Test Set)", fontsize=13, pad=12, weight="bold")
plt.tight_layout()
plt.savefig(fig_dir / "random_forest_confusion_matrix.png", dpi=300)
plt.show()

## 7. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(fpr, tpr, color="#27ae60", lw=2.5, label=f"Random Forest (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], color="#7f8c8d", lw=1.5, linestyle="--", label="Random Guessing (AUC = 0.5000)")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Recall)", fontsize=12)
ax.set_title("Random Forest ROC Curve (Test Set)", fontsize=13, pad=12, weight="bold")
ax.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.savefig(fig_dir / "random_forest_roc_curve.png", dpi=300)
plt.show()

## 8. Feature Importance Extraction & Visualization

In [ ]:
fitted_preprocessor = pipeline.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
importances = pipeline.named_steps["classifier"].feature_importances_

clean_names = [name.replace("num__", "").replace("cat__", "") for name in feature_names]

feat_imp_df = pd.DataFrame({
    "Transformed_Feature": feature_names,
    "Clean_Feature_Name": clean_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

# Export full table to CSV
feat_imp_df.to_csv(output_dir / "random_forest_feature_importance.csv", index=False)
print("Top 10 Feature Importances:")
display(feat_imp_df.head(10))

# Plot top 15 features
top_15 = feat_imp_df.head(15).sort_values(by="Importance", ascending=True)
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(top_15["Clean_Feature_Name"], top_15["Importance"], color="#2ecc71", edgecolor="#27ae60", height=0.65)
ax.set_title("Random Forest: Top 15 Feature Importances", fontsize=14, pad=12, weight="bold")
ax.set_xlabel("Gini Importance Score", fontsize=12)
ax.set_ylabel("Transformed Feature", fontsize=12)
for idx, (val, name) in enumerate(zip(top_15["Importance"], top_15["Clean_Feature_Name"])):
    ax.annotate(f"{val:.4f}", xy=(val, idx), xytext=(5, 0), textcoords="offset points", va="center", fontsize=10, weight="bold")
ax.set_xlim(0, max(top_15["Importance"]) * 1.18)
plt.tight_layout()
plt.savefig(fig_dir / "random_forest_feature_importance.png", dpi=300)
plt.show()

## 9. Save Fitted Pipeline Artifact

In [ ]:
model_path = Path("../models/random_forest_pipeline.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, model_path)
print(f"Pipeline saved to {model_path.resolve()}")